# BÀI THỰC HÀNH 2 - BÀI 6 (TRANG 45)
## SO SÁNH THỜI GIAN XỬ LÝ CỦA APRIORI VÀ FP-GROWTH TRÊN GROCERIES DATASET
- **Học viên:** Nguyễn Minh Vương
- **MSSV:** 6451071090
- **Môn học:** Khai thác dữ liệu
- **Tập dữ liệu:** Groceries Dataset (9835 giao dịch, 169 mặt hàng)

In [ ]:
# Cài đặt thư viện nếu chạy trên Google Colab
# !pip install pandas numpy matplotlib seaborn mlxtend gdown

import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth
%matplotlib inline

# Tự động tải thư mục dataset từ link Google Drive
if not os.path.exists("dataset") or len(os.listdir("dataset")) == 0:
    try:
        import gdown
        drive_folder_url = "https://drive.google.com/drive/folders/1rv5SiO5zFlU-cyEXul09OpCvTXrdjG1m"
        print("Đang tải dataset từ Google Drive...")
        gdown.download_folder(drive_folder_url, output="dataset", quiet=False, use_cookies=False)
        print("✅ Tải dataset thành công!")
    except Exception as e:
        print("Lưu ý: Nếu chưa có thư viện gdown, hãy chạy lệnh: !pip install gdown")

print("Nguyễn Minh Vương - 6451071090")


### 1. Đọc dữ liệu Groceries Dataset và chuyển đổi sang dạng One-hot encoding bằng TransactionEncoder

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

file_path = 'dataset/groceries.csv'

# Đọc từng dòng giao dịch
transactions = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        items = [item.strip() for item in line.strip().split(',') if item.strip()]
        if items:
            transactions.append(items)

print(f"Tổng số giao dịch đã đọc: {len(transactions)}")
print("3 giao dịch đầu tiên:")
for i in range(3):
    print(f"Giao dịch {i+1}: {transactions[i]}")

# Sử dụng TransactionEncoder để biến đổi về ma trận one-hot encoding
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_groceries = pd.DataFrame(te_ary, columns=te.columns_)

print(f"\nKích thước ma trận sau mã hóa: {df_groceries.shape[0]} giao dịch, {df_groceries.shape[1]} mặt hàng")
display(df_groceries.iloc[:5, :10])


### 2. Thực hiện khai thác tập phổ biến với các ngưỡng min_sup khác nhau và ghi nhận thời gian xử lý

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

# Thiết lập dải ngưỡng min_sup để so sánh hiệu năng
min_sup_list = [0.01, 0.02, 0.03, 0.04, 0.05]

time_apriori = []
time_fpgrowth = []
num_itemsets_ap = []
num_itemsets_fp = []

print("Bắt đầu thử nghiệm so sánh hiệu năng giữa Apriori và FP-Growth:\n")

for sup in min_sup_list:
    print(f"--- Đang chạy với min_sup = {sup*100:.1f}% ---")
    
    # 1. Đo thời gian Apriori
    start_ap = time.time()
    itemsets_ap = apriori(df_groceries, min_support=sup, use_colnames=True)
    t_ap = time.time() - start_ap
    time_apriori.append(t_ap)
    num_itemsets_ap.append(len(itemsets_ap))
    print(f"  Apriori:   {t_ap:.4f} giây | Số tập phổ biến: {len(itemsets_ap)}")
    
    # 2. Đo thời gian FP-Growth
    start_fp = time.time()
    itemsets_fp = fpgrowth(df_groceries, min_support=sup, use_colnames=True)
    t_fp = time.time() - start_fp
    time_fpgrowth.append(t_fp)
    num_itemsets_fp.append(len(itemsets_fp))
    print(f"  FP-Growth: {t_fp:.4f} giây | Số tập phổ biến: {len(itemsets_fp)}\n")

# Tổng hợp kết quả so sánh vào bảng
df_compare = pd.DataFrame({
    'Min Support (%)': [s * 100 for s in min_sup_list],
    'Số tập phổ biến': num_itemsets_ap,
    'Thời gian Apriori (s)': time_apriori,
    'Thời gian FP-Growth (s)': time_fpgrowth,
})
df_compare['Tốc độ FP-Growth nhanh hơn (lần)'] = round(df_compare['Thời gian Apriori (s)'] / df_compare['Thời gian FP-Growth (s)'], 2)

print("BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG:")
display(df_compare)


### 3. Vẽ biểu đồ so sánh thời gian xử lý của hai thuật toán và đưa ra nhận xét, giải thích

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

plt.figure(figsize=(10, 6))

plt.plot([s * 100 for s in min_sup_list], time_apriori, marker='o', linewidth=2, color='crimson', label='Apriori')
plt.plot([s * 100 for s in min_sup_list], time_fpgrowth, marker='s', linewidth=2, color='royalblue', label='FP-Growth')

plt.title('So sánh thời gian xử lý giữa Apriori và FP-Growth theo Min Support', fontsize=14)
plt.xlabel('Min Support (%)', fontsize=12)
plt.ylabel('Thời gian thực thi (giây)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.show()


### 4. Nhận xét và giải thích nguyên nhân
> **Nhận xét kết quả:**
> 1. Số lượng tập phổ biến sinh ra từ cả hai thuật toán `Apriori` và `FP-Growth` là hoàn toàn bằng nhau tại mọi ngưỡng `min_sup`.
> 2. Thời gian chạy của thuật toán `FP-Growth` nhanh hơn đáng kể so với `Apriori`. Khi ngưỡng `min_support` càng giảm xuống (ví dụ 1%), sự cách biệt về thời gian càng thể hiện rõ rệt.
>
> **Giải thích nguyên nhân:**
> - **Thuật toán Apriori:** 
>   - Phải trải qua quá trình sinh tập ứng viên (*Candidate Generation*) theo từng bậc $k$. Số lượng ứng viên tăng theo cấp số nhân khi min_sup nhỏ.
>   - Phải quét lại toàn bộ cơ sở dữ liệu sau mỗi lần tăng kích thước tập ứng viên để tính support count, gây nghẽn I/O rất lớn.
> - **Thuật toán FP-Growth:**
>   - Khắc phục hoàn toàn việc sinh tập ứng viên bằng cách nén toàn bộ cơ sở dữ liệu vào một cấu trúc cây tiền tố nhỏ gọn gọi là `FP-tree` (*Frequent Pattern Tree*).
>   - Chỉ cần quét cơ sở dữ liệu đúng **2 lần**: lần 1 để đếm tần số các item đơn lẻ, lần 2 để xây dựng cây FP-tree.
>   - Khai thác tập phổ biến thông qua phân rã đệ quy trên cây con điều kiện (*Conditional FP-tree*), xử lý hoàn toàn trong bộ nhớ RAM nên đạt tốc độ vượt trội.